# Cloud Deployment — Examples

This notebook demonstrates provider-neutral deployment planning, environment configuration, health checks, simple cost estimates, and release validation. The cells run locally and do not create cloud resources.

## 1. Check your local Python environment


In [ ]:
import os, sys, platform

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('APP_ENV:', os.getenv('APP_ENV', 'development'))

## 2. Load application configuration

Only non-secret configuration is printed. For production, retrieve credentials through a managed secret service or workload identity; never commit them to source control.

In [ ]:
from dataclasses import dataclass
import os

@dataclass(frozen=True)
class Settings:
    app_env: str = 'development'
    model_path: str = 'artifacts/model.joblib'
    log_level: str = 'INFO'

def load_settings():
    return Settings(
        app_env=os.getenv('APP_ENV', 'development'),
        model_path=os.getenv('MODEL_PATH', 'artifacts/model.joblib'),
        log_level=os.getenv('LOG_LEVEL', 'INFO').upper(),
    )

settings = load_settings()
settings

## 3. Simulate a health endpoint response

A real FastAPI route could return this payload with HTTP 200 when the service is healthy. Do not expose internal details or secrets.

In [ ]:
def health_response(model_loaded=True):
    if model_loaded:
        return {'http_status': 200, 'body': {'status': 'ok'}}
    return {'http_status': 503, 'body': {'status': 'not_ready'}}

print('Healthy:', health_response(True))
print('Not ready:', health_response(False))

## 4. Build a deployment decision matrix

Use this as a starting point. Scores are illustrative and should be adjusted to your own constraints.

In [ ]:
options = {
    'VM': {'ops_effort': 4, 'flexibility': 5, 'scale_ease': 2},
    'Managed containers': {'ops_effort': 2, 'flexibility': 4, 'scale_ease': 4},
    'Kubernetes': {'ops_effort': 5, 'flexibility': 5, 'scale_ease': 5},
    'Serverless containers': {'ops_effort': 1, 'flexibility': 3, 'scale_ease': 5},
}
print('Higher ops_effort means more operational effort; other scores are relative.')
for name, scores in options.items():
    print(f'{name:22s} {scores}')

## 5. Estimate monthly cost (hypothetical prices)

Replace the hypothetical rates with current provider pricing. This is a planning example, not a quote.

In [ ]:
usage = {
    'compute_hours': 24 * 30,
    'compute_rate_per_hour': 0.08,  # hypothetical currency units/hour
    'storage_gb_month': 20,
    'storage_rate_per_gb_month': 0.03,  # hypothetical
    'egress_gb': 50,
    'egress_rate_per_gb': 0.05,  # hypothetical
    'logging_monthly': 8.00,  # hypothetical
}
costs = {
    'compute': usage['compute_hours'] * usage['compute_rate_per_hour'],
    'storage': usage['storage_gb_month'] * usage['storage_rate_per_gb_month'],
    'egress': usage['egress_gb'] * usage['egress_rate_per_gb'],
    'logging': usage['logging_monthly'],
}
for item, amount in costs.items():
    print(f'{item:10s}: {amount:8.2f}')
print(f"{'TOTAL':10s}: {sum(costs.values()):8.2f}")

## 6. Validate release metrics against example thresholds

The thresholds below are fictional examples. Real thresholds should follow load tests and service-level objectives.

In [ ]:
release_metrics = {'http_5xx_rate': 0.08, 'p95_latency_ms': 920}
example_limits = {'http_5xx_rate': 0.02, 'p95_latency_ms': 500}

breaches = []
for metric, value in release_metrics.items():
    if value > example_limits[metric]:
        breaches.append((metric, value, example_limits[metric]))

if breaches:
    print('Investigate release; consider stopping rollout or rollback:')
    for metric, value, limit in breaches:
        print(f'- {metric}: observed={value}, example_limit={limit}')
else:
    print('No example threshold breaches detected.')

## 7. Generate a deployment checklist

Mark each item only after verifying it for your own service.

In [ ]:
checklist = [
    'Model and preprocessing versions recorded',
    'Tests pass in CI',
    'Container image scanned and tagged immutably',
    'Secrets external to source code and image',
    'Least-privilege service identity configured',
    'TLS and network access controls configured',
    'Health checks, logs, metrics, and alerts configured',
    'Staging smoke test completed',
    'Rollback procedure tested',
    'Costs reviewed and temporary resources cleaned up',
]
for item in checklist:
    print('[ ]', item)

## Further work

Complete `practice.md`, then compare your answers with `solution.md`. For hands-on deployment, first create a minimal API and Dockerfile, then deploy to a staging environment and follow the selected provider's current documentation. This notebook intentionally does not provision cloud resources.